# Analisis Komparatif Algoritma XGBoost, LightGBM, dan CatBoost
## untuk Sistem Deteksi Intrusi pada Dataset NF-UNSW-NB15-v3

Notebook ini dirancang untuk membandingkan tiga model gradient boosting (XGBoost, LightGBM, CatBoost) pada skenario IDS secara **fair** (split data dan preprocessing yang sama).


## 1) Tujuan Komparasi
- Membandingkan performa **XGBoost**, **LightGBM**, dan **CatBoost** pada dataset **NF-UNSW-NB15-v3**.
- Menentukan model terbaik untuk IDS berdasarkan metrik utama.

### Metrik utama
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC
- Waktu latih (training time)
- Waktu inferensi (inference time)


In [ ]:
# 2) Setup Eksperimen (Seed, Environment, Versi Library)
import os
import time
import random
import warnings
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

SEED = 42
TEST_SIZE = 0.2
RANDOM_STATE = SEED

random.seed(SEED)
np.random.seed(SEED)

print("✅ Setup selesai")
print(f"Python      : {platform.python_version()}")
print(f"OS          : {platform.system()} {platform.release()}")
print(f"Seed        : {SEED}")


In [ ]:
# Cek versi library agar eksperimen reproducible
import sklearn
import xgboost
import lightgbm
import catboost

print("Versi library:")
print(f"- pandas    : {pd.__version__}")
print(f"- numpy     : {np.__version__}")
print(f"- scikit-learn: {sklearn.__version__}")
print(f"- xgboost   : {xgboost.__version__}")
print(f"- lightgbm  : {lightgbm.__version__}")
print(f"- catboost  : {catboost.__version__}")


## 3) Load Data Singkat (Minimum)
Notebook akan mencoba menemukan file dataset otomatis dari beberapa lokasi umum.


In [ ]:
# 3.1 Load dataset NF-UNSW-NB15-v3
candidate_paths = [
    Path("NF-UNSW-NB15-v3.csv"),
    Path("./data/NF-UNSW-NB15-v3.csv"),
    Path("./dataset/NF-UNSW-NB15-v3.csv"),
]

# Tambahan: cari file csv dengan nama yang mengandung NF-UNSW-NB15-v3
for p in Path('.').glob("**/*NF-UNSW-NB15-v3*.csv"):
    candidate_paths.append(p)

dataset_path = None
for p in candidate_paths:
    if p.exists() and p.is_file():
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError(
        "Dataset NF-UNSW-NB15-v3 tidak ditemukan. "
        "Letakkan file CSV di root project atau folder data/dataset."
    )

df = pd.read_csv(dataset_path)
print(f"✅ Dataset ditemukan: {dataset_path}")
print(f"Shape data: {df.shape}")

df.head(3)


In [ ]:
# 3.2 Cek ukuran data & distribusi label (imbalanced check)
print("Info dataframe:")
print(df.info())

# Deteksi kolom label secara fleksibel
label_candidates = ["label", "Label", "attack_cat", "attack", "class", "target", "y"]
label_col = next((c for c in label_candidates if c in df.columns), None)

if label_col is None:
    raise ValueError(f"Kolom label tidak ditemukan. Kandidat yang dicek: {label_candidates}")

print(f"\n✅ Kolom label yang digunakan: {label_col}")
print("\nDistribusi label:")
display(df[label_col].value_counts(dropna=False).to_frame("count"))


## 4) Definisi Skenario & Split Data
- Skenario akan otomatis terdeteksi:
  - **Binary** jika jumlah kelas = 2
  - **Multiclass** jika jumlah kelas > 2
- Semua model menggunakan split train/test yang sama (prinsip fairness).


In [ ]:
# 4.1 Pembersihan data minimum + split
# Hapus duplikasi baris (minimum cleaning)
df = df.drop_duplicates().reset_index(drop=True)

X = df.drop(columns=[label_col]).copy()
y_raw = df[label_col].copy()

# Drop kolom ID bila ada (umum pada dataset network flow)
id_like_cols = [c for c in X.columns if c.lower() in {"id", "flow_id", "index"}]
if id_like_cols:
    X = X.drop(columns=id_like_cols)
    print(f"Kolom ID dihapus: {id_like_cols}")

# Encoding label
y_encoder = LabelEncoder()
y = y_encoder.fit_transform(y_raw.astype(str))
class_names = list(y_encoder.classes_)

n_classes = len(np.unique(y))
if n_classes < 3:
    raise ValueError(f"Notebook ini difokuskan untuk multiclass, tetapi hanya terdeteksi {n_classes} kelas.")
scenario = "multiclass"
print(f"Skenario eksperimen: {scenario} ({n_classes} kelas)")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


## 5) Preprocessing Fitur Kategorikal Native untuk Semua Model
- Imputasi missing value numerik (median)
- Imputasi missing value kategorikal (token `MISSING`)
- Menjaga dtype `category` agar bisa dimanfaatkan native oleh model

> Catatan fairness: seluruh model dilatih pada split data yang sama dengan skema imputasi yang sama.


In [ ]:
# 5.1 Preprocessing tanpa one-hot (native categorical) + class weighting multiclass
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

X_train_prep = X_train.copy()
X_test_prep = X_test.copy()

# Imputasi numerik berbasis train
if numeric_cols:
    num_imputer = SimpleImputer(strategy="median")
    X_train_prep[numeric_cols] = num_imputer.fit_transform(X_train_prep[numeric_cols])
    X_test_prep[numeric_cols] = num_imputer.transform(X_test_prep[numeric_cols])

# Imputasi + casting kategorikal ke dtype category agar native categorical tetap aktif
for col in categorical_cols:
    train_col = X_train_prep[col].astype("string").fillna("MISSING")
    test_col = X_test_prep[col].astype("string").fillna("MISSING")

    all_categories = pd.Index(train_col.unique()).union(pd.Index(test_col.unique()))
    X_train_prep[col] = pd.Categorical(train_col, categories=all_categories)
    X_test_prep[col] = pd.Categorical(test_col, categories=all_categories)

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_map = {int(cls): float(w) for cls, w in zip(classes, class_weights)}
sample_weight_train = np.array([class_weight_map[int(lbl)] for lbl in y_train], dtype=float)

print("✅ Preprocessing selesai (native categorical)")
print(f"Jumlah fitur awal        : {X_train.shape[1]}")
print(f"Jumlah fitur kategorikal : {len(categorical_cols)}")
print(f"Jumlah fitur numerik     : {len(numeric_cols)}")
print(f"Contoh class_weight      : {dict(list(class_weight_map.items())[:5])}")


## 6) Definisi Baseline Tiga Model (Multiclass)
- XGBoost dengan `enable_categorical=True`
- LightGBM memanfaatkan dtype `category` dari pandas
- CatBoost menggunakan `cat_features` saat fit


In [ ]:
# 6.1 Baseline model multiclass
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    num_class=n_classes,
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist",
    enable_categorical=True,
)

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    num_leaves=31,
    objective="multiclass",
    num_class=n_classes,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function="MultiClass",
    random_seed=SEED,
    verbose=0,
)

models = {
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model,
    "CatBoost": cat_model,
}

print("✅ Baseline model siap:", list(models.keys()))


## 7) Pelatihan & Prediksi (dengan pencatatan waktu)


In [ ]:
# 7.1 Train + predict + timing
results = []
trained_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    print(f"\n🚀 Training {name}...")

    fit_kwargs = {"sample_weight": sample_weight_train}
    if name == "CatBoost":
        fit_kwargs["cat_features"] = categorical_cols

    t0 = time.perf_counter()
    model.fit(X_train_prep, y_train, **fit_kwargs)
    train_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred = model.predict(X_test_prep)
    infer_time_total = time.perf_counter() - t1
    infer_time_per_sample_ms = (infer_time_total / len(y_test)) * 1000

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test_prep)
    else:
        y_proba = None

    # Metrik klasifikasi multiclass
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    if y_proba is None:
        roc_auc = np.nan
    else:
        roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "Train Time (s)": train_time,
        "Inference Total (s)": infer_time_total,
        "Inference / Sample (ms)": infer_time_per_sample_ms,
    })

    trained_models[name] = model
    predictions[name] = y_pred
    probabilities[name] = y_proba

    print(f"✅ {name} selesai | F1={f1:.4f} | Recall={rec:.4f} | Train={train_time:.3f}s")


## 8) Evaluasi Kuantitatif + Cross-Validation


In [ ]:
# 8.1 Cross-validation multiclass (weighted F1)
cv_rows = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, base_model in models.items():
    fold_scores = []
    for tr_idx, val_idx in cv.split(X_train_prep, y_train):
        X_tr = X_train_prep.iloc[tr_idx].copy()
        X_val = X_train_prep.iloc[val_idx].copy()
        y_tr = y_train[tr_idx]
        y_val = y_train[val_idx]

        fold_classes = np.unique(y_tr)
        fold_weights = compute_class_weight(class_weight="balanced", classes=fold_classes, y=y_tr)
        fold_map = {int(cls): float(w) for cls, w in zip(fold_classes, fold_weights)}
        fold_sample_weight = np.array([fold_map[int(lbl)] for lbl in y_tr], dtype=float)

        model_cv = clone(base_model)
        fit_kwargs = {"sample_weight": fold_sample_weight}
        if name == "CatBoost":
            fit_kwargs["cat_features"] = categorical_cols

        model_cv.fit(X_tr, y_tr, **fit_kwargs)
        y_val_pred = model_cv.predict(X_val)
        fold_f1 = f1_score(y_val, y_val_pred, average="weighted", zero_division=0)
        fold_scores.append(fold_f1)

    cv_rows.append({
        "Model": name,
        "CV F1 (mean)": float(np.mean(fold_scores)),
        "CV F1 (std)": float(np.std(fold_scores)),
    })

cv_results_df = pd.DataFrame(cv_rows)

# Tabel komparasi utama + ranking
results_df = pd.DataFrame(results)
results_df = results_df.merge(cv_results_df, on="Model", how="left")
results_df = results_df.sort_values(by=["F1", "Recall"], ascending=False).reset_index(drop=True)
results_df["Rank (F1->Recall)"] = np.arange(1, len(results_df) + 1)

cols_order = [
    "Rank (F1->Recall)", "Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC",
    "CV F1 (mean)", "CV F1 (std)",
    "Train Time (s)", "Inference Total (s)", "Inference / Sample (ms)"
]
results_df = results_df[cols_order]

print("📊 Tabel Komparasi Utama")
display(results_df)

best_model_name = results_df.iloc[0]["Model"]
print(f"\n🏆 Model terbaik berdasarkan prioritas F1/Recall: {best_model_name}")


In [ ]:
# 8.2 Laporan klasifikasi per model
for name in models.keys():
    print("\n" + "="*80)
    print(f"CLASSIFICATION REPORT - {name}")
    print("="*80)
    print(classification_report(y_test, predictions[name], target_names=[str(c) for c in class_names], zero_division=0))


## 9) Visualisasi Perbandingan
- Bar chart metrik antar model
- Confusion matrix per model
- Grafik waktu komputasi


In [ ]:
# 9.1 Bar chart metrik utama antar model
plot_df = results_df.copy()
plot_df = plot_df.set_index("Model")

metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
ax = plot_df[metrics_to_plot].plot(kind="bar", figsize=(12, 6))
ax.set_title("Perbandingan Metrik Utama Antar Model")
ax.set_ylabel("Skor")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
# 9.2 Confusion matrix per model
n_models = len(models)
fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, name in zip(axes, models.keys()):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=[str(c) for c in class_names],
        yticklabels=[str(c) for c in class_names],
    )
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")

plt.tight_layout()
plt.show()


In [ ]:
# 9.3 Grafik waktu komputasi
time_cols = ["Train Time (s)", "Inference / Sample (ms)"]
ax = results_df.set_index("Model")[time_cols].plot(kind="bar", figsize=(10, 5), color=["#1f77b4", "#ff7f0e"])
ax.set_title("Perbandingan Waktu Komputasi")
ax.set_ylabel("Waktu")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10) Analisis Hasil Komparatif
- Model paling akurat
- Model paling cepat
- Trade-off performa vs efisiensi
- Kesesuaian untuk implementasi IDS real-time


In [ ]:
# 10.1 Ringkasan analisis otomatis
best_accuracy_model = results_df.sort_values("Accuracy", ascending=False).iloc[0]["Model"]
fastest_train_model = results_df.sort_values("Train Time (s)", ascending=True).iloc[0]["Model"]
fastest_infer_model = results_df.sort_values("Inference / Sample (ms)", ascending=True).iloc[0]["Model"]

print("📌 Ringkasan Analisis Komparatif")
print(f"- Model paling akurat              : {best_accuracy_model}")
print(f"- Model training tercepat         : {fastest_train_model}")
print(f"- Model inferensi tercepat        : {fastest_infer_model}")
print(f"- Model terbaik (prioritas F1/Recall): {best_model_name}")

print("\nTrade-off performa vs efisiensi:")
for _, row in results_df.iterrows():
    print(
        f"  • {row['Model']}: F1={row['F1']:.4f}, Recall={row['Recall']:.4f}, "
        f"CV F1={row['CV F1 (mean)']:.4f}±{row['CV F1 (std)']:.4f}, "
        f"Train={row['Train Time (s)']:.3f}s, Infer/sample={row['Inference / Sample (ms)']:.4f}ms"
    )


## 11) Kesimpulan Komparasi
Gunakan sel berikut untuk menghasilkan kesimpulan otomatis berbasis metrik, lalu sesuaikan narasi akhir sesuai konteks penelitian.


In [ ]:
# 11.1 Kesimpulan otomatis
winner = results_df.iloc[0]

print("="*80)
print("KESIMPULAN KOMPARASI")
print("="*80)
print(
    f"Model terbaik pada eksperimen ini adalah {winner['Model']} "
    f"(berdasarkan prioritas F1/Recall)."
)
print(
    f"Nilai utama: Accuracy={winner['Accuracy']:.4f}, Precision={winner['Precision']:.4f}, "
    f"Recall={winner['Recall']:.4f}, F1={winner['F1']:.4f}, ROC-AUC={winner['ROC-AUC']:.4f}."
)
print("\nJustifikasi:")
print("- Pemilihan model didasarkan pada ranking F1 lalu Recall untuk konteks IDS.")
print("- Waktu training dan inferensi turut dipertimbangkan untuk kebutuhan real-time.")

print("\nCatatan keterbatasan komparasi:")
print("- Baseline hyperparameter belum dilakukan tuning ekstensif.")
print("- Variansi antarfold perlu dipertimbangkan bersama metrik hold-out test.")
print("- Hasil sensitif terhadap kualitas data dan distribusi kelas pada NF-UNSW-NB15-v3.")
